<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/nlp-labs/blob/main/Session2/1-Word-Embeddings-copy.ipynb)


# Maestría en Inteligencia Artificial  
## Procesamiento de Lenguaje natural
### Sesión 2 - Práctica

---


**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

# Introducción

# Clasificacion de textos - Analisis de reseñas y/u opiniones

En este notebook se aborda el problema de clasificación de texto aplicado al análisis de reseñas y opiniones en lenguaje natural. El objetivo principal es identificar automáticamente el sentimiento  presente en un conjunto de descripciones textuales utilizando técnicas de representación semántica.

# Configurar entorno

En esta sección se configuran las librerías y dependencias necesarias para el análisis de datos y procesamiento de lenguaje natural. Esto garantiza que el entorno esté listo para cargar, limpiar y analizar las conversaciones del chat político.

In [1]:
import sys
import pkg_resources
import subprocess
import warnings

warnings.filterwarnings('ignore')

# Detectar si estamos en Google Colab
IN_COLAB = 'google.colab' in sys.modules

/tmp/ipykernel_344/3780340132.py:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [12]:
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-r",
        "https://raw.githubusercontent.com/sebastianb92/nlp-labs/main/requirements.txt"
    ])

    # Instala el modelo solo si no existe
    try:
        import spacy
        spacy.load("es_core_news_sm")
    except:
        subprocess.check_call([
            sys.executable, "-m", "spacy", "download", "es_core_news_sm"
        ])
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"
    ])

# 1. Recopilación de datos

Para el presente análisis se utilizará el conjunto de datos mteb/SpanishSentimentClassification, disponible en Hugging Face, el cual contiene reseñas y opiniones en español relacionadas con servicios de hospedaje. Cada registro del dataset se encuentra etiquetado según su polaridad de sentimiento, clasificándose en categorías positivas o negativas.

Este conjunto de datos permitirá evaluar modelos de representación semántica y clasificación automática en una tarea binaria de análisis de sentimiento aplicada a opiniones reales de usuarios.

In [13]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
dataset = load_dataset('sepidmnorozy/Spanish_sentiment', split='train')
# dataset = load_dataset("sentiment140", split="train")
dataset

Dataset({
    features: ['label', 'text'],
    num_rows: 1029
})

In [14]:
dataset[1]

{'label': 1,
 'text': 'un lugar hermosisimo , inolvidable. hermosa atención · rescato el servicio , como asi también la calidez de su gente y el hermoso entorno natural. nunca olvidaremos este viaje. rescato el lugar , la playa. la variedad de comida y la calidad. desde bien tempranito al cruzarte con el personal siempre hay un saludo y buena onda. el tamaño de las habitaciones es espectacular. nos gusto mucho el spa , es un lugar muy familiar y para descansar'}

In [15]:
for i in range(5):
    print(dataset[i])

{'label': 1, 'text': 'Está situado en el centro de la ciudad , con todo lo más turístico a tu alrededor ( El Pilar , por ejemplo ) , y sitios para tomar algo .'}
{'label': 1, 'text': 'un lugar hermosisimo , inolvidable. hermosa atención · rescato el servicio , como asi también la calidez de su gente y el hermoso entorno natural. nunca olvidaremos este viaje. rescato el lugar , la playa. la variedad de comida y la calidad. desde bien tempranito al cruzarte con el personal siempre hay un saludo y buena onda. el tamaño de las habitaciones es espectacular. nos gusto mucho el spa , es un lugar muy familiar y para descansar'}
{'label': 1, 'text': 'Todo absolutamente recomendable .'}
{'label': 1, 'text': 'El metro está a 50 metros bajando tan solo una calle .'}
{'label': 1, 'text': 'Y de Toscana ... mejor que lo veais ... y además delo bonito que es Florencia y Siena ... perderos por sus pueblos ..'}


In [16]:
from collections import Counter
Counter(dataset["label"])

Counter({1: 851, 0: 178})

In [17]:
import numpy as np
np.mean([len(t.split()) for t in dataset["text"]])

np.float64(16.289601554907676)

In [18]:
text_lengths = [len(row['text']) for row in dataset]
print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

Texto más corto: 5
Texto más largo: 608
Longitud promedio: 84.85714285714286
